# 基于MindSpore NLP实现真人照片到特定风格图像生成案例开发

## 项目概览

本案例以开源项目 `cartoonify-main` 为迁移参考，将原始基于 Diffusers 的卡通风格能力整理为一个更适合 **MindSpore 课程应用案例** 的中文 notebook 与交互 DEMO。

案例目标是围绕“**真人照片到风格化人像**”这一应用任务，构建一套可直接运行的推理方案。当前提供三类风格模板：

- **吉卜力动画感**
- **Cartoonify 卡通角色插画感**
- **古风国画感**

整体流程采用“先整图转风格、再局部修人物”的双阶段思路：前者负责风格表达，后者负责身份保持与脸部结构回融。


## 案例介绍

- **从单一卡通模型到多风格应用**：保留 `cartoonify` 的卡通迁移能力，并扩展为吉卜力与古风国画双附加风格。
- **从脚本调用到课程案例**：将原始模型调用方式整理为 notebook 结构，补齐中文说明、参数建议与 DEMO 入口。
- **从单阶段生成到身份保持增强**：增加脸部结构回融、细节恢复与风格后处理，提升人物一致性。
- **从实验界面到展示页**：重新设计交互界面，强调风格卡片、结果展示和参数引导。


## 推荐运行环境

建议环境如下：

| 组件 | 推荐版本 |
| :--- | :--- |
| Python | 3.10.x  |
| MindSpore | 2.7.0 |
| MindSpore NLP | 0.5.1 |
| Gradio | 6.2.0 |
| 运行设备 | Ascend |


## 初始化依赖

本单元负责完成：补充 `mindtorch` 兼容占位；导入推理依赖；


In [ ]:
# -*- coding: utf-8 -*-

try:
    import mindtorch.autograd.function as _mt_function
    if not hasattr(_mt_function, "FunctionCtx"):
        class FunctionCtx:
            pass
        _mt_function.FunctionCtx = FunctionCtx
except Exception as _shim_error:
    print(f"[WARN] mindtorch compatibility shim skipped: {_shim_error}")

import os
import sys
import traceback
from dataclasses import dataclass
from functools import lru_cache
from typing import Dict, Tuple

os.environ.setdefault("HF_HOME", "/root/autodl-tmp/hf_cache")
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/root/autodl-tmp/hf_cache/hub")
os.environ.setdefault("TRANSFORMERS_CACHE", "/root/autodl-tmp/hf_cache/hub")

import numpy as np
from PIL import Image, ImageChops, ImageDraw, ImageEnhance, ImageFilter, ImageOps

import mindspore as ms
import mindnlp
from diffusers import DDIMScheduler, StableDiffusionImg2ImgPipeline
import gradio as gr

INFER_DTYPE = ms.float16
PIPELINE_DEVICE_MAP = "cuda"


## 推理运行参数

为了减少单元执行顺序对 notebook 的影响，推理相关的常量统一放在这一单元中管理，同时打印当前版本信息与缓存路径，便于快速确认环境状态。


In [ ]:
EXPECTED_MS = "2.7.0"
EXPECTED_MNLP = "0.5.1"
EXPECTED_PY_MIN = (3, 10)
EXPECTED_PY_MAX = (3, 12)

print("MindSpore version:", getattr(ms, "__version__", "unknown"))
print("MindSpore NLP version:", getattr(mindnlp, "__version__", "unknown"))
print("Python version:", sys.version.split()[0])
print("device_target:", ms.get_context("device_target"))
print("HF cache:", os.environ.get("HUGGINGFACE_HUB_CACHE"))


## 风格配置中心

这里使用数据类统一描述风格配置，包括：

- 模型地址
- 正向与反向提示词
- 推荐参数
- 界面说明
- 后处理类型

这种写法便于后续继续扩展更多风格，而无需修改主推理流程。


In [ ]:
@dataclass(frozen=True)
class StyleProfile:
    label: str
    model_id: str
    prompt: str
    negative_prompt: str
    recommended_strength: float
    strength_cap_when_preserve: float
    style_face_blend: float
    line_keep: float
    detail_keep: float
    default_steps: int
    default_guidance: float
    intro: str
    recommendation: str
    finish_mode: str
    cartoon_global_boost: float = 0.0
    cartoon_face_boost: float = 0.0


STYLE_LIBRARY: Dict[str, StyleProfile] = {
    "吉卜力(Ghibli)": StyleProfile(
        label="吉卜力(Ghibli)",
        model_id="nitrosocke/Ghibli-Diffusion",
        prompt=(
            "portrait of the same exact person, same identity, same facial proportions, same jawline, same hairstyle, "
            "studio ghibli anime film still, hand-painted anime illustration, clean lineart, soft cel shading, "
            "natural expression, upper body, masterpiece"
        ),
        negative_prompt=(
            "different person, changed face, aged face, child face, huge anime eyes, lowres, blurry, "
            "bad face, deformed face, disfigured, mutated, extra eyes, bad anatomy, watermark, text, logo"
        ),
        recommended_strength=0.40,
        strength_cap_when_preserve=0.48,
        style_face_blend=0.18,
        line_keep=0.22,
        detail_keep=0.28,
        default_steps=25,
        default_guidance=7.5,
        intro="强调手绘动画电影质感，颜色柔和，线稿干净，适合半身人像与轻故事感照片。",
        recommendation="建议从 0.34 ~ 0.44 起步，若想更像本人，优先降低 strength。",
        finish_mode="ghibli",
    ),
    "卡通插画(Cartoon)": StyleProfile(
        label="卡通插画(Cartoon)",
        model_id="lavaman131/cartoonify",
        prompt=(
            "portrait of the same exact person, same identity, same facial proportions, same jawline, same hairstyle, "
            "disney pixar style, polished cartoon illustration, animated feature film character portrait, clean cartoon lineart, "
            "simplified facial planes, soft cel shading, stylized but recognizable face, upper body, masterpiece"
        ),
        negative_prompt=(
            "different person, changed face, exaggerated face, huge eyes, tiny chin, malformed mouth, waxy skin, lowres, blurry, "
            "deformed, bad anatomy, watermark, text, logo"
        ),
        recommended_strength=0.52,
        strength_cap_when_preserve=0.56,
        style_face_blend=0.32,
        line_keep=0.24,
        detail_keep=0.28,
        default_steps=28,
        default_guidance=7.5,
        intro="强调块面与轮廓，卡通化更明显，整体更接近动画角色插画效果。",
        recommendation="建议从 0.46 ~ 0.56 起步，卡通感强但更容易带来五官漂移。",
        finish_mode="cartoon",
        cartoon_global_boost=0.34,
        cartoon_face_boost=0.26,
    ),
    "古风国画(Guohua)": StyleProfile(
        label="古风国画(Guohua)",
        model_id="Langboat/Guohua-Diffusion",
        prompt=(
            "portrait of the same exact person, same identity, same facial proportions, same jawline, same hairstyle, "
            "traditional Chinese guohua painting, elegant ancient Chinese portrait, ink wash painting, refined brush strokes, "
            "soft rice paper texture, graceful costume portrait, artistic composition, masterpiece"
        ),
        negative_prompt=(
            "different person, changed face, modern cartoon, photorealistic, 3d render, over-sharpened, lowres, blurry, "
            "deformed face, bad anatomy, watermark, text, logo"
        ),
        recommended_strength=0.38,
        strength_cap_when_preserve=0.44,
        style_face_blend=0.16,
        line_keep=0.26,
        detail_keep=0.30,
        default_steps=30,
        default_guidance=7.0,
        intro="强调古风人像、笔墨层次与宣纸感，整体更柔和，适合营造东方绘画审美。",
        recommendation="建议从 0.32 ~ 0.42 起步，优先保留面部结构，再逐步增强笔触风格。",
        finish_mode="guohua",
    ),
}

DEFAULT_STYLE = "吉卜力(Ghibli)"


## Pipeline 加载与缓存

为减少重复加载模型造成的时间消耗，这里使用缓存方式管理不同风格对应的 pipeline。


In [ ]:
@lru_cache(maxsize=3)
def get_style_pipeline(model_id: str) -> StableDiffusionImg2ImgPipeline:
    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        model_id,
        ms_dtype=INFER_DTYPE,
        device_map=PIPELINE_DEVICE_MAP,
    )
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

    try:
        pipe.enable_attention_slicing()
    except Exception:
        pass
    try:
        pipe.set_progress_bar_config(disable=True)
    except Exception:
        pass
    try:
        pipe.safety_checker = None
        pipe.requires_safety_checker = False
    except Exception:
        pass

    print(f"[OK] pipeline loaded: {model_id}", flush=True)
    return pipe


## 图像处理与身份保持模块

这一部分是本案例的关键增强点。整体策略为：

1. 先做统一人像预处理；
2. 对整图做一次风格化生成；
3. 对面部区域进行“结构回注”，保留人物身份特征；
4. 根据不同风格再叠加差异化后处理。


In [ ]:
def ensure_rgb_image(image_obj) -> Image.Image:
    if isinstance(image_obj, Image.Image):
        return image_obj.convert("RGB")
    if isinstance(image_obj, ms.Tensor):
        array = np.clip(image_obj.asnumpy(), 0, 255).astype(np.uint8)
        return Image.fromarray(array).convert("RGB")
    if isinstance(image_obj, np.ndarray):
        array = np.clip(image_obj, 0, 255).astype(np.uint8)
        return Image.fromarray(array).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(image_obj)}")


def prepare_portrait(image_obj: Image.Image, size: int) -> Image.Image:
    image_obj = ImageOps.exif_transpose(image_obj).convert("RGB")
    image_obj = ImageEnhance.Sharpness(image_obj).enhance(1.12)
    image_obj = ImageEnhance.Contrast(image_obj).enhance(1.05)
    image_obj = ImageEnhance.Color(image_obj).enhance(1.02)
    return ImageOps.fit(
        image_obj,
        (size, size),
        method=Image.Resampling.LANCZOS,
        centering=(0.5, 0.20),
    )


def estimate_main_face_box(size: int) -> Tuple[int, int, int, int]:
    left = int(size * 0.32)
    top = int(size * 0.11)
    right = int(size * 0.68)
    bottom = int(size * 0.49)
    return left, top, right, bottom


def create_soft_patch_mask(width: int, height: int, blur_radius: int) -> Image.Image:
    mask = Image.new("L", (width, height), 0)
    drawer = ImageDraw.Draw(mask)
    outer = (
        int(width * 0.08),
        int(height * 0.08),
        int(width * 0.92),
        int(height * 0.92),
    )
    inner = (
        int(width * 0.18),
        int(height * 0.14),
        int(width * 0.82),
        int(height * 0.86),
    )
    drawer.rounded_rectangle(outer, radius=max(8, min(width, height) // 9), fill=208)
    drawer.ellipse(inner, fill=255)
    return mask.filter(ImageFilter.GaussianBlur(radius=blur_radius))


def transfer_style_statistics(source_face: Image.Image, target_face: Image.Image) -> Image.Image:
    source = np.asarray(source_face.convert("RGB")).astype(np.float32)
    target = np.asarray(target_face.convert("RGB")).astype(np.float32)
    merged = np.empty_like(source)
    for channel in range(3):
        src = source[..., channel]
        tgt = target[..., channel]
        src_mean, src_std = float(src.mean()), float(src.std()) + 1e-6
        tgt_mean, tgt_std = float(tgt.mean()), float(tgt.std()) + 1e-6
        merged[..., channel] = (src - src_mean) * (tgt_std / src_std) + tgt_mean
    merged = np.clip(merged, 0, 255).astype(np.uint8)
    return Image.fromarray(merged, mode="RGB")


def match_luma_distribution(base_face: Image.Image, style_face: Image.Image) -> Image.Image:
    style_y = style_face.convert("YCbCr").split()[0]
    channels = list(base_face.convert("YCbCr").split())
    channels[0] = Image.blend(channels[0], style_y, 0.35)
    return Image.merge("YCbCr", tuple(channels)).convert("RGB")


def restore_facial_details(base_face: Image.Image, original_face: Image.Image, amount: float) -> Image.Image:
    if amount <= 0:
        return base_face
    sharpened = original_face.filter(ImageFilter.UnsharpMask(radius=1.2, percent=135, threshold=2))
    high_freq = ImageChops.subtract(sharpened, sharpened.filter(ImageFilter.GaussianBlur(radius=1.6)))
    high_freq = ImageOps.autocontrast(high_freq)
    high_freq = ImageEnhance.Contrast(high_freq).enhance(0.82)
    restored = ImageChops.overlay(base_face, high_freq)
    return Image.blend(base_face, restored, float(amount))


def keep_soft_lines(base_face: Image.Image, original_face: Image.Image, amount: float) -> Image.Image:
    if amount <= 0:
        return base_face
    edges = original_face.convert("L").filter(ImageFilter.FIND_EDGES).filter(ImageFilter.GaussianBlur(radius=1.0))
    edges = ImageOps.autocontrast(edges)
    edges = edges.point(lambda pixel: int(255 - pixel * 0.42))
    edge_rgb = Image.merge("RGB", (edges, edges, edges))
    with_lines = ImageChops.multiply(base_face, edge_rgb)
    return Image.blend(base_face, with_lines, float(amount))


def stylize_face_seed(face_image: Image.Image, finish_mode: str) -> Image.Image:
    face_image = face_image.convert("RGB")
    if finish_mode == "cartoon":
        face_image = face_image.filter(ImageFilter.MedianFilter(size=3))
        face_image = face_image.filter(ImageFilter.SMOOTH_MORE)
        face_image = ImageOps.posterize(face_image, 5)
        face_image = ImageEnhance.Color(face_image).enhance(1.10)
        face_image = ImageEnhance.Contrast(face_image).enhance(1.10)
        face_image = ImageEnhance.Sharpness(face_image).enhance(1.18)
        return face_image
    if finish_mode == "guohua":
        face_image = face_image.filter(ImageFilter.SMOOTH_MORE)
        face_image = ImageEnhance.Color(face_image).enhance(0.88)
        face_image = ImageEnhance.Contrast(face_image).enhance(0.94)
        face_image = ImageEnhance.Sharpness(face_image).enhance(0.96)
        return face_image
    face_image = face_image.filter(ImageFilter.SMOOTH)
    face_image = ImageOps.posterize(face_image, 6)
    face_image = ImageEnhance.Color(face_image).enhance(1.04)
    face_image = ImageEnhance.Contrast(face_image).enhance(1.02)
    face_image = ImageEnhance.Sharpness(face_image).enhance(1.06)
    return face_image


def apply_cartoon_finish(image_obj: Image.Image, amount: float) -> Image.Image:
    if amount <= 0:
        return image_obj.convert("RGB")
    base = image_obj.convert("RGB")
    smooth = base.filter(ImageFilter.MedianFilter(size=3)).filter(ImageFilter.SMOOTH_MORE)
    flat = ImageOps.posterize(smooth, 5)
    flat = ImageEnhance.Color(flat).enhance(1.08)
    flat = ImageEnhance.Contrast(flat).enhance(1.10)
    edges = base.convert("L").filter(ImageFilter.FIND_EDGES).filter(ImageFilter.GaussianBlur(radius=0.7))
    edges = ImageOps.autocontrast(edges)
    edges = edges.point(lambda pixel: max(36, 255 - int(pixel * 1.55)))
    edge_rgb = Image.merge("RGB", (edges, edges, edges))
    merged = ImageChops.multiply(flat, edge_rgb)
    merged = ImageEnhance.Sharpness(merged).enhance(1.10)
    return Image.blend(base, merged, float(amount))


def apply_guohua_finish(image_obj: Image.Image) -> Image.Image:
    base = image_obj.convert("RGB")
    softened = base.filter(ImageFilter.GaussianBlur(radius=0.6))
    softened = Image.blend(base, softened, 0.35)
    softened = ImageEnhance.Color(softened).enhance(0.84)
    softened = ImageEnhance.Contrast(softened).enhance(0.93)
    paper = Image.new("RGB", softened.size, (246, 240, 227))
    return Image.blend(paper, softened, 0.82)


def build_identity_patch(original_face: Image.Image, stylized_face: Image.Image, profile: StyleProfile) -> Image.Image:
    remapped = transfer_style_statistics(original_face, stylized_face)
    remapped = match_luma_distribution(remapped, stylized_face)
    seeded = stylize_face_seed(remapped, profile.finish_mode)
    fused = Image.blend(seeded, stylized_face, float(profile.style_face_blend))
    fused = restore_facial_details(fused, original_face, float(profile.detail_keep))
    fused = keep_soft_lines(fused, original_face, float(profile.line_keep))
    if profile.finish_mode == "cartoon":
        fused = apply_cartoon_finish(fused, amount=float(profile.cartoon_face_boost))
        fused = ImageEnhance.Color(fused).enhance(1.05)
        fused = ImageEnhance.Contrast(fused).enhance(1.08)
        fused = ImageEnhance.Sharpness(fused).enhance(1.20)
    elif profile.finish_mode == "guohua":
        fused = apply_guohua_finish(fused)
        fused = ImageEnhance.Sharpness(fused).enhance(0.96)
    else:
        fused = ImageEnhance.Color(fused).enhance(1.02)
        fused = ImageEnhance.Sharpness(fused).enhance(1.10)
    return fused


## 主推理流程

主流程包含两个阶段：

1. **整图风格化**：使用目标风格模型完成全图 `img2img` 推理；
2. **局部身份回注**：在面部区域回注原始结构，减少“重新捏脸”的问题。


In [ ]:
def run_img2img(
    pipe: StableDiffusionImg2ImgPipeline,
    prompt: str,
    negative_prompt: str,
    image_obj: Image.Image,
    strength: float,
    steps: int,
    guidance_scale: float,
):
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image_obj,
        strength=float(strength),
        num_inference_steps=int(steps),
        guidance_scale=float(guidance_scale),
    )
    if hasattr(result, "images") and result.images:
        return result.images[0]
    if isinstance(result, (list, tuple)) and result:
        return result[0]
    return result


def stylize_portrait(
    image: Image.Image,
    style_name: str,
    strength: float,
    steps: int,
    guidance_scale: float,
    seed: int,
    size: int,
    preserve_identity: bool,
) -> Image.Image:
    if image is None:
        raise ValueError("请先上传一张真人照片。")
    if style_name not in STYLE_LIBRARY:
        raise ValueError(f"暂不支持风格：{style_name}")

    profile = STYLE_LIBRARY[style_name]
    pipe = get_style_pipeline(profile.model_id)
    prepared = prepare_portrait(image, size=size)

    if int(seed) > 0:
        ms.set_seed(int(seed))
        np.random.seed(int(seed))

    global_strength = min(float(strength), float(profile.strength_cap_when_preserve)) if preserve_identity else float(strength)

    global_result = ensure_rgb_image(
        run_img2img(
            pipe=pipe,
            prompt=profile.prompt,
            negative_prompt=profile.negative_prompt,
            image_obj=prepared,
            strength=global_strength,
            steps=int(steps),
            guidance_scale=float(guidance_scale),
        )
    )

    if profile.finish_mode == "cartoon":
        global_result = apply_cartoon_finish(global_result, amount=float(profile.cartoon_global_boost))
    elif profile.finish_mode == "guohua":
        global_result = apply_guohua_finish(global_result)

    if not preserve_identity:
        return global_result

    face_box = estimate_main_face_box(size)
    origin_face = prepared.crop(face_box)
    stylized_face = global_result.crop(face_box)
    patched_face = build_identity_patch(origin_face, stylized_face, profile)

    patch_width = face_box[2] - face_box[0]
    patch_height = face_box[3] - face_box[1]
    patched_face = patched_face.resize((patch_width, patch_height), Image.Resampling.LANCZOS)
    face_mask = create_soft_patch_mask(patch_width, patch_height, blur_radius=max(3, size // 128))

    merged = global_result.copy()
    merged.paste(patched_face, (face_box[0], face_box[1]), mask=face_mask)
    return merged


## 风格联动说明

为了减少用户试错，界面在切换风格时会自动刷新：

- 风格介绍
- 推荐调参建议
- 默认 `strength`
- 默认 `steps`
- 默认 `guidance_scale`


In [ ]:
def get_style_panel(style_name: str):
    profile = STYLE_LIBRARY[style_name]
    info_text = (
        f"### {profile.label}\n"
        f"**风格特点：** {profile.intro}\n\n"
        f"**调参建议：** {profile.recommendation}"
    )
    return (
        info_text,
        profile.recommended_strength,
        profile.default_steps,
        profile.default_guidance,
    )


def generate_for_ui(img, style, strength, steps, guidance, seed, size, preserve_identity):
    try:
        result = stylize_portrait(
            image=img,
            style_name=style,
            strength=float(strength),
            steps=int(steps),
            guidance_scale=float(guidance),
            seed=int(seed),
            size=int(size),
            preserve_identity=bool(preserve_identity),
        )
        return result
    except Exception as error:
        traceback.print_exc()
        raise gr.Error(str(error))


APP_DESCRIPTION = '''
# 真人照片到特定风格图像生成DEMO
支持吉卜力、Cartoonify 卡通插画、古风国画三类效果。
'''


## 交互 DEMO 布局

界面层重新组织为三部分：

- 顶部展示项目说明与使用建议；
- 中间采用“左参数、右结果”的双栏布局；
- 风格卡片、参数区、结果区彼此分组，界面更接近成品演示页。


In [ ]:
CUSTOM_CSS = '''
.demo-shell {max-width: 1180px; margin: 0 auto;}
.hero-card {
    padding: 4px 0 6px 0;
    border-radius: 0;
    background: transparent;
    border: none;
    box-shadow: none;
}
.panel-note {
    border-radius: 14px;
    padding: 10px 14px;
    background: #f7f1e7;
    border: 1px solid #eadfce;
}
.compact-gap {gap: 8px;}
.generate-btn button {
    background: linear-gradient(135deg, #c96a3d 0%, #a84a2a 100%) !important;
    border: none !important;
    color: white !important;
    box-shadow: 0 10px 24px rgba(169, 74, 42, 0.22) !important;
}
.generate-btn button:hover {
    filter: brightness(1.04);
}
.gradio-container h1, .gradio-container h2, .gradio-container h3 {
    letter-spacing: 0.02em;
}
'''

with gr.Blocks(title="Photo2Style DEMO", css=CUSTOM_CSS) as demo:
    with gr.Column(elem_classes=["demo-shell"]):
        with gr.Group(elem_classes=["hero-card"]):
            gr.Markdown(APP_DESCRIPTION)

        with gr.Row(equal_height=True, elem_classes=["compact-gap"]):
            with gr.Column(scale=4):
                gr.Markdown("## 控制台")
                style_name = gr.Dropdown(
                    choices=list(STYLE_LIBRARY.keys()),
                    value=DEFAULT_STYLE,
                    label="选择风格模板",
                )
                style_card = gr.Markdown()

                with gr.Group():
                    input_image = gr.Image(type="pil", label="上传原始照片")
                    preserve_identity = gr.Checkbox(
                        value=True,
                        label="人物特征保留增强",
                    )
                    size = gr.Dropdown(
                        choices=[512, 640, 768],
                        value=512,
                        label="输出尺寸",
                    )
                    strength = gr.Slider(
                        minimum=0.20,
                        maximum=0.75,
                        value=STYLE_LIBRARY[DEFAULT_STYLE].recommended_strength,
                        step=0.01,
                        label="风格强度（strength）",
                    )
                    steps = gr.Slider(
                        minimum=10,
                        maximum=50,
                        value=STYLE_LIBRARY[DEFAULT_STYLE].default_steps,
                        step=1,
                        label="推理步数（steps）",
                    )
                    guidance = gr.Slider(
                        minimum=1.0,
                        maximum=12.0,
                        value=STYLE_LIBRARY[DEFAULT_STYLE].default_guidance,
                        step=0.5,
                        label="文本引导强度（guidance_scale）",
                    )
                    seed = gr.Number(value=0, precision=0, label="随机种子（0 表示随机）")
                    run_button = gr.Button(
                        "生成风格化结果",
                        variant="primary",
                        size="lg",
                        elem_classes=["generate-btn"],
                    )

            with gr.Column(scale=5):
                gr.Markdown("## 结果展示")
                with gr.Row(elem_classes=["compact-gap"]):
                    output_image = gr.Image(type="pil", label="生成结果", height=560)

                gr.Markdown(
                    "<div class='panel-note'>"
                    "<b>结果说明：</b> 若出现风格很强但不像本人，可降低 strength；"
                    "若风格表达不明显，可适当提高 steps 或 strength。"
                    "</div>"
                )

        with gr.Row():
            gr.Markdown(
                "### 使用提示\n"
                "- Cartoonify 风格更适合中高强度参数\n"
                "- 吉卜力更适合较柔和的强度范围\n"
                "- 古风国画建议从低强度开始，逐步增强笔触感"
            )

    demo.load(
        fn=lambda: get_style_panel(DEFAULT_STYLE),
        outputs=[style_card, strength, steps, guidance],
    )
    style_name.change(
        fn=get_style_panel,
        inputs=[style_name],
        outputs=[style_card, strength, steps, guidance],
    )
    run_button.click(
        fn=generate_for_ui,
        inputs=[input_image, style_name, strength, steps, guidance, seed, size, preserve_identity],
        outputs=[output_image],
    )

demo


## 启动 DEMO

执行下方单元即可启动 Gradio 服务。如需自定义端口，可修改环境变量 `PORT` 或直接修改 `server_port`。


In [ ]:
demo.queue(max_size=20).launch(
    server_name="0.0.0.0",
    server_port=int(os.getenv("PORT", "7861")),
    show_error=True,
)

## 后续可扩展方向

- 接入更多风格模型，例如钢笔画、工笔画、迪士尼角色风。
- 引入更准确的人脸检测与多人脸选择策略。
- 将 DEMO 部署到魔乐社区，并在此处补充正式访问链接。

> DEMO 链接：待部署后补充。
